# NarrativeNexus: Dynamic Text Analysis Platform

This notebook implements a comprehensive text analysis pipeline organized by weekly milestones.


## Setup and Installation


In [ ]:
# Install required packages
%pip install pandas nltk scikit-learn transformers torch textblob wordcloud matplotlib seaborn plotly -q


In [ ]:
# Import all necessary libraries
import os
import re
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize, sent_tokenize
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
from textblob import TextBlob
from transformers import pipeline
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Download NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print("✓ All libraries loaded successfully")


---
## Week 1: Data Collection and Input Handling


In [ ]:
def load_text_data(file_path):
    """
    Reads text content from a file with error handling.
    
    Args:
        file_path: Path to the text file
    
    Returns:
        str: File content or None if error occurs
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read().strip()
        
        if len(content) == 0:
            print(f"Warning: File {file_path} is empty")
            return None
        
        print(f"✓ Successfully loaded: {os.path.basename(file_path)}")
        print(f"  File size: {len(content)} characters")
        return content
    
    except FileNotFoundError:
        print(f"✗ Error: File not found at {file_path}")
        return None
    except Exception as e:
        print(f"✗ Error reading file: {e}")
        return None

def validate_text_input(text):
    """
    Validates that the input text meets minimum requirements.
    """
    if not text:
        return False, "Text is empty"
    
    if len(text) < 50:
        return False, "Text is too short (minimum 50 characters required)"
    
    word_count = len(text.split())
    if word_count < 10:
        return False, "Text has too few words (minimum 10 words required)"
    
    return True, f"Validation passed: {word_count} words found"

# Load the sample text file
input_file = "Sample_text.txt"
raw_text = load_text_data(input_file)

if raw_text:
    is_valid, message = validate_text_input(raw_text)
    print(f"\n{message}")
    if is_valid:
        print("\n--- Text Preview (first 200 characters) ---")
        print(raw_text[:200] + "...")
    else:
        print(f"\n✗ Validation failed: {message}")
        raw_text = None


In [ ]:
def clean_text_data(text):
    """
    Comprehensive text cleaning function.
    Removes URLs, hashtags, usernames, special characters, and normalizes text.
    """
    if not text:
        return ""
    
    # Remove URLs (http, https, www)
    text = re.sub(r'https?://\S+|www\.\S+', '', text, flags=re.IGNORECASE)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove hashtags and mentions
    text = re.sub(r'#\w+|@\w+', '', text)
    
    # Remove copyright symbols and special markers
    text = re.sub(r'©|®|™', '', text)
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

def tokenize_and_lemmatize(text):
    """
    Tokenizes text and applies lemmatization.
    Removes stopwords and non-alphabetic tokens.
    """
    # Tokenize
    tokens = word_tokenize(text)
    
    # Get stopwords
    stop_words = set(stopwords.words('english'))
    
    # Initialize lemmatizer
    lemmatizer = WordNetLemmatizer()
    
    # Process tokens: filter, lemmatize, and keep only alphabetic words
    processed_tokens = []
    for token in tokens:
        # Keep only alphabetic tokens with length > 2
        if token.isalpha() and len(token) > 2:
            if token not in stop_words:
                lemmatized = lemmatizer.lemmatize(token)
                processed_tokens.append(lemmatized)
    
    return processed_tokens

def prepare_text_for_analysis(raw_text):
    """
    Main preprocessing pipeline.
    Returns both cleaned text (for sentiment) and processed tokens (for topic modeling).
    """
    # Step 1: Clean text
    cleaned = clean_text_data(raw_text)
    
    # Step 2: Tokenize and lemmatize
    tokens = tokenize_and_lemmatize(cleaned)
    
    # Step 3: Join tokens back for vectorization
    processed_text = ' '.join(tokens)
    
    return cleaned, processed_text, tokens

# Apply preprocessing
if raw_text:
    cleaned_text, processed_text, token_list = prepare_text_for_analysis(raw_text)
    
    print("✓ Preprocessing completed")
    print(f"  Original length: {len(raw_text)} characters")
    print(f"  Cleaned length: {len(cleaned_text)} characters")
    print(f"  Processed tokens: {len(token_list)} words")
    print(f"\n--- Sample processed tokens (first 20) ---")
    print(token_list[:20])
else:
    print("✗ Cannot proceed: No valid text data available")


In [ ]:
def extract_topics_lda(text, num_topics=5, num_words=8):
    """
    Extracts topics using Latent Dirichlet Allocation (LDA).
    """
    if not text or len(text.split()) < num_topics:
        return [], "Insufficient text for topic modeling"
    
    # Create document list (single document in this case)
    documents = [text]
    
    # Vectorize using TF-IDF
    vectorizer = TfidfVectorizer(
        max_df=0.95,
        min_df=1,
        max_features=1000,
        stop_words='english'
    )
    
    try:
        tfidf_matrix = vectorizer.fit_transform(documents)
        
        # Adjust number of topics if needed
        actual_topics = min(num_topics, tfidf_matrix.shape[1])
        
        if actual_topics < 2:
            return [], "Not enough unique terms for topic modeling"
        
        # Train LDA model
        lda_model = LatentDirichletAllocation(
            n_components=actual_topics,
            random_state=42,
            max_iter=10,
            learning_method='online'
        )
        
        lda_model.fit(tfidf_matrix)
        
        # Extract topic words
        feature_names = vectorizer.get_feature_names_out()
        topics = []
        
        for topic_idx, topic in enumerate(lda_model.components_):
            top_word_indices = topic.argsort()[-num_words:][::-1]
            top_words = [feature_names[i] for i in top_word_indices]
            topics.append(top_words)
        
        return topics, "Success"
    
    except Exception as e:
        return [], f"Error in LDA: {str(e)}"

def extract_topics_nmf(text, num_topics=5, num_words=8):
    """
    Extracts topics using Non-Negative Matrix Factorization (NMF).
    """
    if not text or len(text.split()) < num_topics:
        return [], "Insufficient text for topic modeling"
    
    documents = [text]
    
    # Use CountVectorizer for NMF
    vectorizer = CountVectorizer(
        max_df=0.95,
        min_df=1,
        max_features=1000,
        stop_words='english'
    )
    
    try:
        count_matrix = vectorizer.fit_transform(documents)
        
        actual_topics = min(num_topics, count_matrix.shape[1])
        
        if actual_topics < 2:
            return [], "Not enough unique terms for topic modeling"
        
        # Train NMF model
        nmf_model = NMF(
            n_components=actual_topics,
            random_state=42,
            max_iter=200
        )
        
        nmf_model.fit(count_matrix)
        
        # Extract topic words
        feature_names = vectorizer.get_feature_names_out()
        topics = []
        
        for topic_idx, topic in enumerate(nmf_model.components_):
            top_word_indices = topic.argsort()[-num_words:][::-1]
            top_words = [feature_names[i] for i in top_word_indices]
            topics.append(top_words)
        
        return topics, "Success"
    
    except Exception as e:
        return [], f"Error in NMF: {str(e)}"

# Apply topic modeling
if processed_text:
    print("=== LDA Topic Modeling ===")
    lda_topics, lda_status = extract_topics_lda(processed_text, num_topics=5, num_words=7)
    
    if lda_topics:
        print(f"✓ LDA completed: {len(lda_topics)} topics identified")
        for i, topic in enumerate(lda_topics, 1):
            print(f"  Topic {i}: {', '.join(topic)}")
    else:
        print(f"✗ LDA failed: {lda_status}")
    
    print("\n=== NMF Topic Modeling ===")
    nmf_topics, nmf_status = extract_topics_nmf(processed_text, num_topics=5, num_words=7)
    
    if nmf_topics:
        print(f"✓ NMF completed: {len(nmf_topics)} topics identified")
        for i, topic in enumerate(nmf_topics, 1):
            print(f"  Topic {i}: {', '.join(topic)}")
    else:
        print(f"✗ NMF failed: {nmf_status}")
    
    # Use LDA topics as primary (or NMF if LDA failed)
    selected_topics = lda_topics if lda_topics else nmf_topics
else:
    selected_topics = []
    print("✗ Cannot perform topic modeling: No processed text available")


In [ ]:
def analyze_sentiment(text):
    """
    Performs sentiment analysis using TextBlob.
    Returns polarity score and sentiment label.
    """
    if not text:
        return 0.0, "Neutral", "No text provided"
    
    blob = TextBlob(text)
    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity
    
    # Classify sentiment
    if polarity > 0.15:
        sentiment_label = "Positive"
        emoji = "😊"
    elif polarity < -0.15:
        sentiment_label = "Negative"
        emoji = "😞"
    else:
        sentiment_label = "Neutral"
        emoji = "😐"
    
    report = f"Sentiment: {sentiment_label} {emoji}\n"
    report += f"Polarity Score: {polarity:.3f} (range: -1.0 to +1.0)\n"
    report += f"Subjectivity: {subjectivity:.3f} (range: 0.0 to 1.0)"
    
    return polarity, sentiment_label, report

def analyze_sentence_sentiments(text):
    """
    Analyzes sentiment for each sentence to understand distribution.
    """
    sentences = sent_tokenize(text)
    sentence_sentiments = []
    
    for sentence in sentences:
        if len(sentence.strip()) > 10:  # Skip very short sentences
            blob = TextBlob(sentence)
            polarity = blob.sentiment.polarity
            sentence_sentiments.append(polarity)
    
    return sentence_sentiments

# Perform sentiment analysis
if cleaned_text:
    polarity_score, sentiment_label, sentiment_report = analyze_sentiment(cleaned_text)
    sentence_polarities = analyze_sentence_sentiments(cleaned_text)
    
    print("=== Overall Sentiment Analysis ===")
    print(sentiment_report)
    
    if sentence_polarities:
        print(f"\n=== Sentence-level Analysis ===")
        print(f"Total sentences analyzed: {len(sentence_polarities)}")
        print(f"Average sentence polarity: {np.mean(sentence_polarities):.3f}")
        print(f"Positive sentences: {sum(1 for p in sentence_polarities if p > 0.15)}")
        print(f"Negative sentences: {sum(1 for p in sentence_polarities if p < -0.15)}")
        print(f"Neutral sentences: {sum(1 for p in sentence_polarities if -0.15 <= p <= 0.15)}")
else:
    polarity_score = 0.0
    sentiment_label = "Neutral"
    sentence_polarities = []
    print("✗ Cannot perform sentiment analysis: No cleaned text available")


---
## Week 5: Summarization and Insights Generation


In [ ]:
def extractive_summarization(text, num_sentences=3):
    """
    Extractive summarization by selecting top sentences based on TF-IDF scores.
    """
    sentences = sent_tokenize(text)
    
    if len(sentences) <= num_sentences:
        return ' '.join(sentences)
    
    # Calculate TF-IDF for sentences
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(sentences)
    
    # Calculate sentence scores
    sentence_scores = tfidf_matrix.sum(axis=1).A1
    
    # Get top N sentences
    top_indices = sentence_scores.argsort()[-num_sentences:][::-1]
    top_sentences = [sentences[i] for i in sorted(top_indices)]
    
    return ' '.join(top_sentences)

def abstractive_summarization(text):
    """
    Abstractive summarization using transformer models.
    """
    try:
        # Truncate text to model's maximum length
        max_length = 1024
        truncated_text = text[:max_length] if len(text) > max_length else text
        
        # Use a lightweight summarization model
        summarizer = pipeline(
            "summarization",
            model="sshleifer/distilbart-cnn-12-6",
            device=-1  # Use CPU
        )
        
        summary = summarizer(
            truncated_text,
            max_length=120,
            min_length=40,
            do_sample=False
        )
        
        return summary[0]['summary_text']
    
    except Exception as e:
        return f"Could not generate abstractive summary: {str(e)}"

def generate_actionable_insights(polarity, topics, sentence_sentiments):
    """
    Generates actionable insights and recommendations based on analysis results.
    """
    insights = []
    
    # Sentiment-based insights
    if polarity < -0.15:
        insights.append("⚠️ Negative sentiment detected. Immediate attention required.")
        insights.append("Recommendation: Investigate pain points and address customer concerns.")
    elif polarity > 0.15:
        insights.append("✓ Positive sentiment identified. Opportunity to leverage strengths.")
        insights.append("Recommendation: Highlight positive aspects in marketing and communication.")
    else:
        insights.append("➖ Neutral sentiment observed. Mixed feedback detected.")
        insights.append("Recommendation: Conduct deeper analysis to understand specific opinions.")
    
    # Topic-based insights
    if topics:
        primary_topic = topics[0][:5] if len(topics) > 0 else []
        if primary_topic:
            insights.append(f"\n🔍 Primary themes identified: {', '.join(primary_topic)}")
            insights.append("Recommendation: Focus analysis on these key areas.")
    
    # Sentiment distribution insights
    if sentence_sentiments:
        positive_count = sum(1 for p in sentence_sentiments if p > 0.15)
        negative_count = sum(1 for p in sentence_sentiments if p < -0.15)
        
        if positive_count > negative_count:
            insights.append(f"\n📊 Sentiment distribution: {positive_count} positive vs {negative_count} negative statements.")
        elif negative_count > positive_count:
            insights.append(f"\n📊 Sentiment distribution: {negative_count} negative vs {positive_count} positive statements.")
            insights.append("Recommendation: Prioritize addressing negative feedback.")
    
    return '\n'.join(insights)

# Generate summaries and insights
if cleaned_text:
    print("=== Extractive Summary ===")
    extractive_summary = extractive_summarization(cleaned_text, num_sentences=4)
    print(extractive_summary)
    
    print("\n=== Abstractive Summary ===")
    abstractive_summary = abstractive_summarization(cleaned_text)
    print(abstractive_summary)
    
    print("\n=== Actionable Insights ===")
    insights = generate_actionable_insights(polarity_score, selected_topics, sentence_polarities)
    print(insights)
else:
    extractive_summary = ""
    abstractive_summary = ""
    insights = ""
    print("✗ Cannot generate summaries: No text data available")


---
## Week 6-7: Visualization and Reporting


In [ ]:
def create_word_cloud(text):
    """
    Generates a word cloud visualization.
    """
    if not text or len(text.split()) < 5:
        print("Insufficient text for word cloud")
        return
    
    try:
        wordcloud = WordCloud(
            width=1000,
            height=500,
            background_color='white',
            colormap='viridis',
            max_words=100,
            relative_scaling=0.5
        ).generate(text)
        
        plt.figure(figsize=(14, 7))
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.axis('off')
        plt.title('Key Themes Word Cloud', fontsize=16, pad=20)
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Error generating word cloud: {e}")

def plot_sentiment_analysis(polarity, sentence_polarities):
    """
    Creates sentiment visualization charts.
    """
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Overall Sentiment Score', 'Sentence-level Sentiment Distribution'),
        specs=[[{"type": "bar"}, {"type": "histogram"}]]
    )
    
    # Overall sentiment bar
    color = 'green' if polarity > 0.15 else ('red' if polarity < -0.15 else 'gray')
    fig.add_trace(
        go.Bar(
            x=[polarity],
            y=['Sentiment'],
            orientation='h',
            marker=dict(color=color),
            name='Polarity Score'
        ),
        row=1, col=1
    )
    
    # Sentence distribution histogram
    if sentence_polarities:
        fig.add_trace(
            go.Histogram(
                x=sentence_polarities,
                nbinsx=20,
                marker=dict(color='steelblue'),
                name='Sentence Sentiments'
            ),
            row=1, col=2
        )
    
    fig.update_xaxes(title_text="Polarity Score", range=[-1, 1], row=1, col=1)
    fig.update_xaxes(title_text="Polarity Score", range=[-1, 1], row=1, col=2)
    fig.update_yaxes(title_text="", row=1, col=1)
    fig.update_yaxes(title_text="Frequency", row=1, col=2)
    
    fig.update_layout(
        title_text="Sentiment Analysis Dashboard",
        height=500,
        showlegend=False
    )
    
    fig.show()

def plot_topic_distribution(topics):
    """
    Visualizes topic distribution.
    """
    if not topics:
        print("No topics available for visualization")
        return
    
    # Count words per topic
    topic_sizes = [len(topic) for topic in topics]
    topic_labels = [f"Topic {i+1}" for i in range(len(topics))]
    
    fig = go.Figure(data=[
        go.Bar(
            x=topic_labels,
            y=topic_sizes,
            marker=dict(color='lightblue', line=dict(color='navy', width=1.5)),
            text=[f"{len(t)} words" for t in topics],
            textposition='outside'
        )
    ])
    
    fig.update_layout(
        title="Topic Distribution",
        xaxis_title="Topics",
        yaxis_title="Number of Key Words",
        height=400
    )
    
    fig.show()

# Generate visualizations
if processed_text:
    print("=== Generating Visualizations ===\n")
    
    print("1. Word Cloud")
    create_word_cloud(processed_text)
    
    print("\n2. Sentiment Analysis Charts")
    plot_sentiment_analysis(polarity_score, sentence_polarities)
    
    print("\n3. Topic Distribution")
    plot_topic_distribution(selected_topics)
else:
    print("✗ Cannot generate visualizations: No processed data available")


In [ ]:
def generate_comprehensive_report():
    """
    Generates a comprehensive analysis report.
    """
    report = []
    report.append("=" * 60)
    report.append("NARRATIVENEXUS: COMPREHENSIVE TEXT ANALYSIS REPORT")
    report.append("=" * 60)
    report.append("")
    
    # File information
    report.append("📄 FILE INFORMATION")
    report.append("-" * 60)
    report.append(f"Source File: {input_file}")
    if raw_text:
        report.append(f"Original Text Length: {len(raw_text)} characters")
        report.append(f"Word Count: {len(raw_text.split())} words")
    report.append("")
    
    # Preprocessing summary
    report.append("🔧 PREPROCESSING SUMMARY")
    report.append("-" * 60)
    if processed_text:
        report.append(f"Processed Tokens: {len(token_list)} words")
        report.append(f"Cleaning: URLs, hashtags, usernames removed")
        report.append(f"Normalization: Lowercasing, lemmatization applied")
    report.append("")
    
    # Topic modeling results
    report.append("🔬 TOPIC MODELING RESULTS")
    report.append("-" * 60)
    if selected_topics:
        report.append(f"Number of Topics Identified: {len(selected_topics)}")
        for i, topic in enumerate(selected_topics, 1):
            report.append(f"  Topic {i}: {', '.join(topic[:5])}")
    else:
        report.append("No topics could be extracted.")
    report.append("")
    
    # Sentiment analysis
    report.append("📊 SENTIMENT ANALYSIS")
    report.append("-" * 60)
    report.append(f"Overall Sentiment: {sentiment_label}")
    report.append(f"Polarity Score: {polarity_score:.3f}")
    if sentence_polarities:
        report.append(f"Sentences Analyzed: {len(sentence_polarities)}")
    report.append("")
    
    # Summaries
    report.append("✨ SUMMARIES")
    report.append("-" * 60)
    report.append("Extractive Summary:")
    report.append(extractive_summary[:200] + "..." if len(extractive_summary) > 200 else extractive_summary)
    report.append("")
    report.append("Abstractive Summary:")
    report.append(abstractive_summary)
    report.append("")
    
    # Insights
    report.append("💡 ACTIONABLE INSIGHTS")
    report.append("-" * 60)
    report.append(insights)
    report.append("")
    
    report.append("=" * 60)
    report.append("Report generated by NarrativeNexus Platform")
    report.append("=" * 60)
    
    return '\n'.join(report)

# Generate and display report
if raw_text:
    final_report = generate_comprehensive_report()
    print(final_report)
    
    # Optionally save to file
    output_file = "analysis_report.txt"
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(final_report)
    print(f"\n✓ Report saved to {output_file}")
else:
    print("✗ Cannot generate report: No data available")


---
## Summary

This notebook implements the complete NarrativeNexus text analysis pipeline:

- ✅ **Week 1**: Data collection and validation
- ✅ **Week 2**: Text preprocessing and normalization
- ✅ **Week 3**: Topic modeling (LDA and NMF)
- ✅ **Week 4**: Sentiment analysis
- ✅ **Week 5**: Summarization and insights generation
- ✅ **Week 6-7**: Visualizations and reporting
- ✅ **Week 8**: Comprehensive report generation

All analysis results are available in the variables above and can be exported or further processed as needed.
